In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))  # Add parent folder

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.competitors import (
    export_benchmark_matrices,
    get_rezaeian_features,
    map_competitor_features,
)
from src.utils import load_metabric_data

ModuleNotFoundError: No module named 'src'

In [ ]:
# Load and Filter for Luminal A
df = load_metabric_data()
df_LumA = df[df['pam50_+_claudin-low_subtype'] == 'LumA'].copy()
df_LumA['target_mortality'] = df_LumA['death_from_cancer'].apply(
    lambda x: 1 if str(x).strip().lower() == 'died of disease' else 0)

In [ ]:
df_LumA.shape

In [ ]:
df_LumA.info()

In [ ]:
'integrative_cluster' in df_LumA.columns

In [ ]:
df_LumA['integrative_cluster'].value_counts()

In [ ]:
'abc' in 'abcd'

In [ ]:
# Map Competitor Features
rezaeian_list = get_rezaeian_features()
features = map_competitor_features(df_LumA.columns, rezaeian_list)

In [ ]:
rezaeian_list

In [ ]:
X = df_LumA[features].copy()
y = df_LumA['target_mortality']

In [ ]:
X.info()

In [ ]:
X['smad2_mut'].value_counts()

In [ ]:
for col in X.columns:
    if col.endswith('_mut'):
        # Convert mutation strings to binary (0 or 1)
        X[col] = (X[col] != '0').astype(int)
    else:
        # Ensure mRNA values are numeric
        X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

In [ ]:
X['smad2_mut'].value_counts()

In [ ]:
# Split & Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Replicate the Grid Search from original benchmark
models_to_test = {
    'LogisticRegression': (LogisticRegression(max_iter=1000, random_state=42), {
        'C': [0.1, 1, 10], 'class_weight': ['balanced', None]
    }),
    'SVM': (SVC(probability=True, random_state=42), {
        'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear'], 'class_weight': ['balanced', None]
    }),
    'RandomForest': (RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 10], 'class_weight': ['balanced', None]
    })
}

best_bench_model = None
best_score = 0
winner_name = ""

for name, (model, params) in models_to_test.items():
    grid = GridSearchCV(model, params, cv=5, scoring='roc_auc')
    grid.fit(X_train_scaled, y_train)
    if grid.best_score_ > best_score:
        best_score = grid.best_score_
        best_bench_model = grid.best_estimator_
        winner_name = name

In [ ]:
best_bench_model

In [ ]:
# Demonstrate WITHOUT tuning
export_benchmark_matrices(best_bench_model, X_test_scaled, y_test,
                          threshold=0.5, suffix=f"{winner_name}_Untuned")

In [ ]:
# Demonstrate WITH tuning (PR-Optimal)
export_benchmark_matrices(best_bench_model, X_test_scaled, y_test,
                          threshold=None, suffix=f"{winner_name}_Tuned")

In [ ]:
print("WITH THRESHOLD TUNING (Precision-Recall Optimal)")
# The function automatically calculates the best PR threshold if we don't force one
export_benchmark_matrices(
    best_bench_model,
    X_test_scaled,
    y_test,
    suffix="Tuned_PR_Curve",
    save_file=False
)